# Estimating Population and Demographic Characteristics in Nigeria Using Google Colab

## Seven-Chapter Student Notebook

Choose a Nigerian State/FCT and LGA/Area Council, then progressively analyse mapped settlements, population, sex composition, children under 1 and under 5, health facilities, roads, drainage, and population density.

### Where your work is saved
This notebook mounts **your own Google Drive**. Final outputs are saved under:

`MyDrive/Nigeria_Population_Demography_Course/Student_Work/<State>/<LGA>/Final_Results/`

Final deliverables are **CSV tables and PNG maps only**.

**Scientific rules:** population values are modelled estimates; GRID3 settlement extents are mapped extents rather than official community boundaries; facility presence does not prove operational status; under-1 is already included in under-5.

Run the notebook from top to bottom.




# Chapter 1 — Choose and map your study LGA

Select a State/FCT and LGA/Area Council from the validated SALB administrative lookup. Stable IDs are `adm1cd` and `adm2cd`.




In [ ]:
# Cell 1 — Install tools, mount Google Drive, and create your personal course folder
!pip -q install geopandas pyogrio shapely pyproj rasterio exactextract requests pyarrow fiona s2sphere ipywidgets folium osm2geojson mapclassify adjustText

from pathlib import Path
import json, re, subprocess, sys, requests
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# Mount the student's own Google Drive.
drive.mount("/content/drive")

REPO="zubairgis/nigeria-population-demography-course"
RAW=f"https://raw.githubusercontent.com/{REPO}/main"

# Temporary processing area. This may disappear when the Colab runtime resets.
TEMP_ROOT=Path("/content/nigeria_demography_course")
TEMP_ROOT.mkdir(parents=True,exist_ok=True)

# Persistent student area. Files here remain in the student's Google Drive.
DRIVE_ROOT=Path("/content/drive/MyDrive/Nigeria_Population_Demography_Course/Student_Work")
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)

states=pd.read_csv(f"{RAW}/data/lookups/admin_states_source.csv",dtype=str)
lgas=pd.read_csv(f"{RAW}/data/lookups/admin_lgas_source.csv",dtype=str)

assert len(states)==37 and len(lgas)==774
print("Google Drive mounted successfully.")
print("Persistent course folder:", DRIVE_ROOT)
print("State/FCT:",len(states)," LGA/Area Council:",len(lgas))






In [ ]:
# Cell 2 — Choose State/FCT and LGA
state_dropdown=widgets.Dropdown(options=sorted(states.adm1nm.unique()),value="Bayelsa",description="State/FCT:")
lga_dropdown=widgets.Dropdown(description="LGA:")
def refresh(*_):
    opts=sorted(lgas.loc[lgas.adm1nm.eq(state_dropdown.value),"adm2nm"].unique())
    lga_dropdown.options=opts
    lga_dropdown.value="Sagbama" if state_dropdown.value=="Bayelsa" and "Sagbama" in opts else opts[0]
state_dropdown.observe(refresh,names="value"); refresh()
display(widgets.VBox([state_dropdown,lga_dropdown]))
print("Choose above, then run Cell 3.")


In [ ]:
# Cell 3 — Load selected SALB boundary, create your personal work folder, and map it
SELECTED_STATE=state_dropdown.value
SELECTED_LGA=lga_dropdown.value

STATE_ID=states.loc[states.adm1nm.eq(SELECTED_STATE),"adm1cd"].iloc[0]
safe=lambda s: re.sub(r"[^A-Za-z0-9]+","_",str(s)).strip("_")

# Persistent folder for this student's selected study area.
STUDENT_WORK_DIR=DRIVE_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
STUDENT_WORK_DIR.mkdir(parents=True,exist_ok=True)

# Temporary processing folder for this run.
WORK=TEMP_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
WORK.mkdir(parents=True,exist_ok=True)

url=f"{RAW}/data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state/{STATE_ID}_{safe(SELECTED_STATE)}.geojson"
state_gdf=gpd.read_file(url).to_crs(4326)
selected=state_gdf[state_gdf.adm2nm.astype(str).eq(SELECTED_LGA)].copy()

if len(selected)!=1:
    raise ValueError("Selected LGA was not uniquely resolved.")

LGA_ID=str(selected.iloc[0].adm2cd)

# Save the boundary only in the temporary processing folder.
# Final student deliverables are CSV and PNG only.
LGA_FILE=WORK/f"{LGA_ID}.geojson"
selected.to_file(LGA_FILE,driver="GeoJSON")

# Remove any boundary GeoJSON left by an older notebook run.
for old_boundary in STUDENT_WORK_DIR.glob("*_boundary.geojson"):
    old_boundary.unlink()

BBOX=tuple(float(x) for x in selected.total_bounds)
geom=selected.geometry.iloc[0]
center=[geom.centroid.y,geom.centroid.x]

# Download OpenStreetMap roads for the selected LGA.
# Public Overpass servers can occasionally be busy, so try several endpoints.
import osm2geojson
from shapely.geometry import shape

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
]

minx,miny,maxx,maxy=BBOX
overpass_query = f"""
[out:json][timeout:120];
(
  way["highway"]({miny},{minx},{maxy},{maxx});
);
out geom;
"""

osm_json = None
last_error = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        rr = requests.post(
            endpoint,
            data={"data": overpass_query},
            timeout=150,
            headers={"User-Agent":"Hensard-University-Nigeria-Demography-Course/1.0"}
        )
        rr.raise_for_status()
        osm_json = rr.json()
        print("OSM roads source:", endpoint)
        break
    except Exception as e:
        last_error = e
        print("Overpass endpoint unavailable:", endpoint)

if osm_json is None:
    print("OSM roads could not be downloaded; continuing without roads.")
    OSM_ROADS = gpd.GeoDataFrame(
        columns=["highway","name","geometry"], geometry="geometry", crs=4326
    )
else:
    roads_geojson = osm2geojson.json2geojson(osm_json)
    OSM_ROADS = gpd.GeoDataFrame.from_features(roads_geojson["features"], crs=4326)
    if not OSM_ROADS.empty:
        OSM_ROADS = OSM_ROADS[
            OSM_ROADS.geometry.geom_type.isin(["LineString","MultiLineString"])
        ].copy()
        OSM_ROADS = gpd.clip(OSM_ROADS, selected)
        keep_cols=[c for c in ["highway","name","geometry"] if c in OSM_ROADS.columns]
        OSM_ROADS=OSM_ROADS[keep_cols]
    print("OSM road features inside selected LGA:", len(OSM_ROADS))

# Blank vector map: no satellite or place-name basemap.
m=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)


folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature: {
        "fillColor": "#4da3ff",
        "color": "#0057b8",
        "weight": 3,
        "fillOpacity": 0.18
    },
    tooltip=folium.Tooltip(f"{SELECTED_STATE} — {SELECTED_LGA}")
).add_to(m)

if len(OSM_ROADS)>0:
    folium.GeoJson(
        OSM_ROADS.__geo_interface__,
        name="OSM Roads",
        style_function=lambda feature: {
            "color": "#ffd54f",
            "weight": 2,
            "opacity": 0.9
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[f for f in ["name","highway"] if f in OSM_ROADS.columns],
            aliases=[f"{f.title()}:" for f in ["name","highway"] if f in OSM_ROADS.columns]
        ) if any(f in OSM_ROADS.columns for f in ["name","highway"]) else None
    ).add_to(m)

minx,miny,maxx,maxy=BBOX
m.fit_bounds([[miny,minx],[maxy,maxx]])
folium.LayerControl(collapsed=False).add_to(m)
display(m)

print("State ID:",STATE_ID)
print("LGA ID:",LGA_ID)
print("Persistent student folder:",STUDENT_WORK_DIR)








# Chapter 2 — Discover settlements and settlement names

Use GRID3 NGA Settlement Extents v3.1 and GRID3 Settlement Names. Zero-name and multiple-name cases remain explicit rather than being forced.




In [ ]:
# Cell 4 — Retrieve settlement extents and use GRID3 Settlement Names cached in GitHub
REPO_DIR=WORK/"repo"
if REPO_DIR.exists():
    subprocess.run(["rm","-rf",str(REPO_DIR)],check=True)

subprocess.run(
    ["git","clone","--depth","1",f"https://github.com/{REPO}.git",str(REPO_DIR)],
    check=True
)

VEC=WORK/"vectors"
VEC.mkdir(exist_ok=True)
bbox=[str(v) for v in BBOX]

# -----------------------------------------------------------------
# OSM DRAINAGE — prefer the State/FCT cache stored in GitHub
# -----------------------------------------------------------------
DRAINAGE_CACHE_FILE = (
    REPO_DIR/"data/drainage/by_state"/
    f"{STATE_ID}_{safe(SELECTED_STATE)}_drainage.geojson"
)

if DRAINAGE_CACHE_FILE.exists():
    DRAINAGE = gpd.read_file(DRAINAGE_CACHE_FILE, bbox=BBOX).to_crs(4326)
    if len(DRAINAGE):
        DRAINAGE = gpd.clip(DRAINAGE, selected)
    print("Drainage source: GitHub State/FCT OSM cache")
else:
    # Fallback so the notebook remains usable while the repository cache is building.
    minx,miny,maxx,maxy=BBOX
    drainage_query=f"""
    [out:json][timeout:120];
    (
      way["waterway"~"^(river|stream|canal|drain)$"]({miny},{minx},{maxy},{maxx});
    );
    out geom;
    """
    drainage_json=None
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            rr=requests.post(
                endpoint,
                data={"data":drainage_query},
                timeout=150,
                headers={"User-Agent":"Hensard-University-Nigeria-Demography-Course/1.0"}
            )
            rr.raise_for_status()
            drainage_json=rr.json()
            print("Temporary drainage source:",endpoint)
            break
        except Exception as e:
            print("Drainage endpoint unavailable:",endpoint)

    if drainage_json is None:
        DRAINAGE=gpd.GeoDataFrame(
            columns=["waterway","name","geometry"],
            geometry="geometry",crs=4326
        )
    else:
        drainage_geojson=osm2geojson.json2geojson(drainage_json)
        DRAINAGE=gpd.GeoDataFrame.from_features(
            drainage_geojson["features"],crs=4326
        )
        if len(DRAINAGE):
            DRAINAGE=DRAINAGE[
                DRAINAGE.geometry.geom_type.isin(["LineString","MultiLineString"])
            ].copy()
            keep=[c for c in ["waterway","name","geometry"] if c in DRAINAGE.columns]
            DRAINAGE=DRAINAGE[keep]
            DRAINAGE=gpd.clip(DRAINAGE,selected)

print("Drainage features inside selected LGA:",len(DRAINAGE))
if len(DRAINAGE) and "waterway" in DRAINAGE.columns:
    display(DRAINAGE["waterway"].value_counts())

# 1. GRID3 Settlement Extents v3.1:
#    downloaded from the versioned GitHub Release configured by the repository.
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","settlement_extents",
    "--output",str(VEC),
    "--bbox",*bbox
],check=True)

# 2. GRID3 Settlement Names:
#    read ONLY from the copy cached in this GitHub repository.
NAMES_ROOT=REPO_DIR/"data/settlement_names"
NAMES_MANIFEST=NAMES_ROOT/"manifest.json"

if not NAMES_MANIFEST.exists():
    raise FileNotFoundError(
        "The GitHub GRID3 Settlement Names cache is not available yet. "
        "Wait for the 'Cache GRID3 settlement names' GitHub Action to finish, "
        "then restart this notebook from Cell 1."
    )

names_meta=json.loads(NAMES_MANIFEST.read_text())
storage_mode=names_meta["storage_mode"]

if storage_mode=="national":
    names_source=NAMES_ROOT/"GRID3_NGA_settlement_names.geojson"
    grid3_names=gpd.read_file(names_source,bbox=BBOX).to_crs(4326)

elif storage_mode=="by_state":
    parts=[]
    for item in names_meta["files"]:
        p=NAMES_ROOT/item["file"]
        try:
            x=gpd.read_file(p,bbox=BBOX)
            if len(x)>0:
                parts.append(x)
        except Exception:
            pass
    if parts:
        grid3_names=gpd.GeoDataFrame(
            pd.concat(parts,ignore_index=True),
            geometry="geometry",
            crs=parts[0].crs
        ).to_crs(4326)
    else:
        grid3_names=gpd.GeoDataFrame(
            columns=["set_id","set_name","geometry"],
            geometry="geometry",
            crs=4326
        )
else:
    raise ValueError(f"Unknown GRID3 settlement-name storage mode: {storage_mode}")

# Keep only points spatially relevant to the selected LGA.
if len(grid3_names)>0:
    grid3_names=gpd.clip(grid3_names,selected)

GRID3_NAMES_FILE=VEC/"settlement_names.geojson"
grid3_names.to_file(GRID3_NAMES_FILE,driver="GeoJSON")

print("GRID3 settlement-name points from GitHub inside selected LGA:",len(grid3_names))
if len(grid3_names)>0 and "set_name" in grid3_names.columns:
    display(
        grid3_names[
            [c for c in ["set_id","set_name","set_altnam","lganame","geometry"]
             if c in grid3_names.columns]
        ].head(20)
    )

# 3. Match cached GRID3 names to GRID3 v3.1 settlement extents.
SETT=WORK/"settlement_components.gpkg"

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/04_prepare_settlements.py"),
    "--extents",str(VEC/"settlement_extents.geojson"),
    "--names",str(GRID3_NAMES_FILE),
    "--lga",str(LGA_FILE),
    "--lga-id",LGA_ID,
    "--out",str(SETT)
],check=True)

settlements=gpd.read_file(SETT).to_crs(4326)

print("Settlement components:",len(settlements))
print("Name matching:")
display(settlements.name_match_status.value_counts(dropna=False))

# 4. Vector-only preview map: no satellite imagery or external place labels.
sett_map=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)

folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature:{
        "fillColor":"#4da3ff",
        "color":"#00b0ff",
        "weight":3,
        "fillOpacity":0.08
    }
).add_to(sett_map)

ROAD_PREVIEW_STYLE={
    "motorway":("#d73027",5),
    "trunk":("#f46d43",4),
    "primary":("#fdae61",3.5),
    "secondary":("#fee08b",3),
    "tertiary":("#969696",2.4),
    "residential":("#bdbdbd",1.7),
    "unclassified":("#cccccc",1.5),
    "service":("#d9d9d9",1.2)
}

if len(OSM_ROADS)>0:
    folium.GeoJson(
        OSM_ROADS.__geo_interface__,
        name="OSM Roads",
        style_function=lambda feature:{
            "color":ROAD_PREVIEW_STYLE.get(
                str(feature["properties"].get("highway","unclassified")),
                ("#cccccc",1.2)
            )[0],
            "weight":ROAD_PREVIEW_STYLE.get(
                str(feature["properties"].get("highway","unclassified")),
                ("#cccccc",1.2)
            )[1],
            "opacity":0.9
        }
    ).add_to(sett_map)

DRAIN_PREVIEW_STYLE={
    "river":("#2166ac",4),
    "stream":("#4393c3",2.5),
    "canal":("#67a9cf",2.2),
    "drain":("#92c5de",1.5)
}
if len(DRAINAGE)>0:
    folium.GeoJson(
        DRAINAGE.__geo_interface__,
        name="Drainage",
        style_function=lambda feature:{
            "color":DRAIN_PREVIEW_STYLE.get(
                str(feature["properties"].get("waterway","stream")),
                ("#4393c3",2)
            )[0],
            "weight":DRAIN_PREVIEW_STYLE.get(
                str(feature["properties"].get("waterway","stream")),
                ("#4393c3",2)
            )[1],
            "opacity":0.9
        }
    ).add_to(sett_map)

if len(grid3_names)>0:
    for _,r in grid3_names.iterrows():
        nm=str(r.get("set_name","")).strip()
        if nm:
            folium.Marker(
                [r.geometry.y,r.geometry.x],
                icon=folium.DivIcon(
                    html=f'<div style="font-size:10px;font-weight:600;color:#1b5e20;'
                         f'white-space:nowrap;">{nm}</div>'
                )
            ).add_to(sett_map)

minx,miny,maxx,maxy=BBOX
sett_map.fit_bounds([[miny,minx],[maxy,maxx]])
folium.LayerControl(collapsed=False).add_to(sett_map)
display(sett_map)





# Chapter 3 — Estimate total population\n\nUse the **WorldPop v2 API** for the selected LGA. No WorldPop GeoTIFF is downloaded or stored.\n



In [ ]:
# Cell 5 — Set up the WorldPop v2 API and choose an indicator
import time
from shapely.geometry import mapping

WORLDPOP_API = "https://api.worldpop.org/v2"
WORLDPOP_YEAR = 2025
WORLDPOP_RESOLUTION = "100m"

# WorldPop v2 accepts Polygon or MultiPolygon geometry in WGS84.
LGA_GEOMETRY = mapping(selected.to_crs(4326).geometry.iloc[0])

def worldpop_submit(endpoint, payload, poll_seconds=2, timeout_seconds=180):
    """Submit a WorldPop v2 task and wait for the result."""
    r = requests.post(f"{WORLDPOP_API}/{endpoint}", json=payload, timeout=60)
    r.raise_for_status()
    submitted = r.json()
    task_id = submitted.get("task_id")
    if not task_id:
        raise RuntimeError(f"WorldPop did not return a task_id: {submitted}")

    started = time.time()
    while True:
        check = requests.get(f"{WORLDPOP_API}/tasks/{task_id}", timeout=60)
        check.raise_for_status()
        info = check.json()

        status = str(info.get("status","")).lower()
        if status == "success":
            return info["result"]
        if status in {"failure","failed","error"}:
            raise RuntimeError(f"WorldPop task failed: {info.get('error') or info}")

        if time.time() - started > timeout_seconds:
            raise TimeoutError(f"WorldPop task {task_id} exceeded {timeout_seconds} seconds.")
        time.sleep(poll_seconds)

def query_population(geometry):
    return worldpop_submit("population", {
        "geojson": geometry,
        "year": WORLDPOP_YEAR,
        "resolution": WORLDPOP_RESOLUTION,
    })

def query_agesex(geometry, age_range=(0,90), sex="both"):
    return worldpop_submit("agesex", {
        "geojson": geometry,
        "year": WORLDPOP_YEAR,
        "age_range": list(age_range),
        "sex": sex,
        "resolution": WORLDPOP_RESOLUTION,
    })

indicator_dropdown = widgets.Dropdown(
    options=[
        ("Total population","total"),
        ("Male population","male"),
        ("Female population","female"),
        ("Children under 1","u1"),
        ("Children under 5","u5"),
    ],
    value="total",
    description="Indicator:"
)
display(indicator_dropdown)

print("WorldPop API configured for:", WORLDPOP_YEAR, WORLDPOP_RESOLUTION)
print("No raster file will be downloaded or stored.")


In [ ]:
# Cell 6 — Get total population for the selected LGA from WorldPop API
pop_result = query_population(LGA_GEOMETRY)

population_total_est = float(pop_result["total_population"])

print(f"{SELECTED_LGA}, {SELECTED_STATE}")
print(f"Estimated total population ({WORLDPOP_YEAR}): {population_total_est:,.0f}")
print("WorldPop source:", pop_result.get("data_source"))
print("Area (km²):", pop_result.get("area_km2"))

# Start the small LGA summary table used by later chapters.
lga_pop = pd.DataFrame([{
    "adm2cd": LGA_ID,
    "population_total_est": population_total_est,
    "population_reference_year": WORLDPOP_YEAR,
    "population_dataset_id": "WorldPop Global 2 via API v2",
    "resolution": WORLDPOP_RESOLUTION,
}])
display(lga_pop)


# Chapter 4 — Describe male and female population\n\nUse the WorldPop age-sex API to obtain the male and female population for the selected LGA.\n



In [ ]:
# Cell 7 — Get male and female population from WorldPop API
agesex_all = query_agesex(LGA_GEOMETRY, age_range=(0,90), sex="both")
pyramid = pd.DataFrame(agesex_all["agesex_pyramid"])

population_male_est = float(pd.to_numeric(pyramid["male"], errors="coerce").sum())
population_female_est = float(pd.to_numeric(pyramid["female"], errors="coerce").sum())

lga_pop["population_male_est"] = population_male_est
lga_pop["population_female_est"] = population_female_est

sex = pd.DataFrame({
    "group":["Male","Female"],
    "estimate":[population_male_est,population_female_est]
})
display(sex)

ax=sex.plot.bar(
    x="group", y="estimate", legend=False,
    title=f"{SELECTED_LGA}: modelled male and female population"
)
ax.set_ylabel("Estimated people")

print("Male + female:", f"{population_male_est + population_female_est:,.0f}")
print("API total:", f"{population_total_est:,.0f}")


# Chapter 5 — Identify children under 1 and under 5\n\nUse the WorldPop age-sex API for ages 0–4. Under-1 is included within under-5. An optional settlement explorer lets students query one mapped settlement at a time without downloading rasters.\n



In [ ]:
# Cell 8 — Get children under 1 and under 5 from WorldPop API
# Request only ages 0–4. The returned classes include age 0 and ages 1–4.
agesex_u5 = query_agesex(LGA_GEOMETRY, age_range=(0,4), sex="both")
u5_pyramid = pd.DataFrame(agesex_u5["agesex_pyramid"])

# WorldPop classes are normally "00" for age 0–1 and "01" for ages 1–5.
u5_pyramid["class"] = u5_pyramid["class"].astype(str).str.zfill(2)
u5_pyramid["both"] = (
    pd.to_numeric(u5_pyramid["male"], errors="coerce").fillna(0)
    + pd.to_numeric(u5_pyramid["female"], errors="coerce").fillna(0)
)

u1_rows = u5_pyramid[u5_pyramid["class"].eq("00")]
u1 = float(u1_rows["both"].sum())
u5 = float(u5_pyramid["both"].sum())

if not (0 <= u1 <= u5 <= population_total_est * 1.01):
    raise ValueError("Unexpected demographic hierarchy returned by the API.")

lga_pop["population_u1_est"] = u1
lga_pop["population_u5_est"] = u5

child_summary = pd.DataFrame({
    "indicator":["Under 1","Under 5"],
    "estimate":[u1,u5],
    "percent_total":[100*u1/population_total_est,100*u5/population_total_est]
})
display(child_summary.round(2))
display(u5_pyramid)

print("Under-1 is included within under-5; do not add it again.")


In [ ]:
# Cell 9 — Optional API explorer: choose one settlement and one indicator
# This intentionally queries ONE settlement at a time to avoid hundreds of API jobs.

settlement_options = []
for i,row in settlements.reset_index(drop=True).iterrows():
    name = row.get("settlement_name")
    if pd.isna(name) or str(name).strip()=="":
        label = f"Unnamed settlement component {i+1}"
    else:
        label = str(name)
    settlement_options.append((f"{label} [{row['settlement_component_id']}]", row["settlement_component_id"]))

settlement_dropdown = widgets.Dropdown(
    options=settlement_options,
    description="Settlement:",
    layout=widgets.Layout(width="95%")
)
sett_indicator_dropdown = widgets.Dropdown(
    options=[
        ("Total population","total"),
        ("Male population","male"),
        ("Female population","female"),
        ("Children under 1","u1"),
        ("Children under 5","u5"),
    ],
    value=indicator_dropdown.value,
    description="Indicator:"
)

display(widgets.VBox([settlement_dropdown,sett_indicator_dropdown]))
print("Choose a settlement and indicator above, then run Cell 10.")


In [ ]:
# Cell 10 — Query the selected settlement/indicator
chosen_id = settlement_dropdown.value
chosen_indicator = sett_indicator_dropdown.value
chosen = settlements[settlements["settlement_component_id"].astype(str).eq(str(chosen_id))].copy()

if len(chosen) != 1:
    raise ValueError("Selected settlement component was not uniquely resolved.")

chosen_geom = mapping(chosen.to_crs(4326).geometry.iloc[0])

if chosen_indicator == "total":
    result = query_population(chosen_geom)
    value = float(result["total_population"])

else:
    if chosen_indicator in {"u1","u5"}:
        result = query_agesex(chosen_geom, age_range=(0,4), sex="both")
    else:
        result = query_agesex(chosen_geom, age_range=(0,90), sex="both")

    p = pd.DataFrame(result["agesex_pyramid"])
    p["class"] = p["class"].astype(str).str.zfill(2)
    male = pd.to_numeric(p["male"], errors="coerce").fillna(0)
    female = pd.to_numeric(p["female"], errors="coerce").fillna(0)

    if chosen_indicator == "male":
        value = float(male.sum())
    elif chosen_indicator == "female":
        value = float(female.sum())
    elif chosen_indicator == "u1":
        value = float((male + female)[p["class"].eq("00")].sum())
    elif chosen_indicator == "u5":
        value = float((male + female).sum())

settlement_api_result = pd.DataFrame([{
    "settlement_component_id": chosen_id,
    "settlement_name": chosen.iloc[0].get("settlement_name"),
    "indicator": chosen_indicator,
    "estimate": value,
    "reference_year": WORLDPOP_YEAR,
    "source": "WorldPop API v2"
}])

display(settlement_api_result)
print("No raster was downloaded.")


In [ ]:
# Cell 11 — Estimate settlement population density with the WorldPop API
# The API returns total_population, area_km2 and population_density for each polygon.
# Results are cached as CSV in Google Drive so reruns can resume without repeating completed queries.

from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import mapping
import time

DENSITY_CACHE = STUDENT_WORK_DIR / "settlement_population_density_cache.csv"

# Query named settlement components. Unnamed polygons remain visible on maps but are
# left as No data for settlement-density interpretation.
density_base = settlements.copy()
density_base["settlement_name"] = density_base.get(
    "settlement_name", pd.Series(index=density_base.index, dtype="object")
)
named_mask = density_base["settlement_name"].notna() & density_base["settlement_name"].astype(str).str.strip().ne("")
query_settlements = density_base.loc[named_mask].copy()

# Load completed API results from an earlier run, if available.
if DENSITY_CACHE.exists():
    density_cache = pd.read_csv(DENSITY_CACHE, dtype={"settlement_component_id": str})
else:
    density_cache = pd.DataFrame(columns=[
        "settlement_component_id","settlement_name",
        "population_total_est","area_km2",
        "population_density_p_km2","api_status"
    ])

done_ids = set(density_cache.loc[
    density_cache["api_status"].eq("success"),
    "settlement_component_id"
].astype(str))

pending = query_settlements[
    ~query_settlements["settlement_component_id"].astype(str).isin(done_ids)
].copy()

print("Named settlement polygons:", len(query_settlements))
print("Already cached:", len(done_ids))
print("Still to query:", len(pending))

# Safety: unauthenticated WorldPop API use has a documented daily request limit.
# A very large LGA should be processed in more than one session rather than silently
# exceeding the public service limit.
MAX_NEW_POLYGONS_PER_RUN = 400
if len(pending) > MAX_NEW_POLYGONS_PER_RUN:
    print(
        f"Processing the first {MAX_NEW_POLYGONS_PER_RUN} pending settlements in this run. "
        "Rerun this cell later to continue from the CSV cache."
    )
    pending = pending.head(MAX_NEW_POLYGONS_PER_RUN)

def _query_one_settlement(row):
    sid = str(row["settlement_component_id"])
    sname = row.get("settlement_name")
    try:
        result = query_population(mapping(row.geometry))
        return {
            "settlement_component_id": sid,
            "settlement_name": sname,
            "population_total_est": float(result.get("total_population", np.nan)),
            "area_km2": float(result.get("area_km2", np.nan)),
            "population_density_p_km2": float(result.get("population_density", np.nan)),
            "api_status": "success"
        }
    except Exception as e:
        return {
            "settlement_component_id": sid,
            "settlement_name": sname,
            "population_total_est": np.nan,
            "area_km2": np.nan,
            "population_density_p_km2": np.nan,
            "api_status": f"failed: {type(e).__name__}"
        }

new_rows = []
if len(pending) > 0:
    # A small worker pool keeps the lesson practical without aggressively loading
    # the public API.
    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = [ex.submit(_query_one_settlement, row) for _,row in pending.iterrows()]
        for n,fut in enumerate(as_completed(futures), start=1):
            new_rows.append(fut.result())
            if n % 20 == 0 or n == len(futures):
                print(f"Completed {n}/{len(futures)} new settlement queries.")

if new_rows:
    density_cache = pd.concat(
        [density_cache, pd.DataFrame(new_rows)],
        ignore_index=True
    )
    density_cache = density_cache.drop_duplicates(
        subset=["settlement_component_id"],
        keep="last"
    )
    density_cache.to_csv(DENSITY_CACHE, index=False)

# Merge cached results into the settlement polygons used for mapping.
settlements["settlement_component_id"] = settlements["settlement_component_id"].astype(str)
density_cache["settlement_component_id"] = density_cache["settlement_component_id"].astype(str)

settlements = settlements.drop(
    columns=[c for c in [
        "population_total_est","area_km2","population_density_p_km2","api_status"
    ] if c in settlements.columns]
).merge(
    density_cache.drop(columns=["settlement_name"], errors="ignore"),
    on="settlement_component_id",
    how="left",
    validate="one_to_one"
)

print(
    "Settlement polygons with population density:",
    int(settlements["population_density_p_km2"].notna().sum()),
    "of", len(settlements)
)

display(
    density_cache[density_cache["api_status"].eq("success")]
    .sort_values("population_density_p_km2", ascending=False)
    .head(20)
)




# Chapter 6 — Locate health facilities and explore roads and drainage

Use GRID3/NHFR health-facility records, OpenStreetMap road classes, and the repository-cached OpenStreetMap drainage network for the selected LGA.



In [ ]:
# Cell 12 — Prepare health facilities inside the selected LGA
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","health_v2",
    "--output",str(VEC),
    "--bbox",*bbox
],check=True)

HEALTH=WORK/"health.gpkg"

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/06_prepare_health_facilities.py"),
    "--facilities",str(VEC/"health_v2.geojson"),
    "--lga",str(LGA_FILE),
    "--out",str(HEALTH)
],check=True)

health=gpd.read_file(HEALTH).to_crs(4326)

print("Health facilities inside selected LGA:",len(health))

if len(health)>0:
    display(
        health[
            [c for c in [
                "facility_name",
                "ownership",
                "facility_level",
                "latitude",
                "longitude"
            ] if c in health.columns]
        ].head(20)
    )




# Chapter 7 — Reconcile, interpret and export

Build final demographic summaries, check consistency, and export professional **CSV tables and PNG maps**. Final maps use vector cartography with selected-LGA boundary, surrounding-LGA labels only, settlement fills without outlines, categorized roads, drainage, GRID3 settlement names, health-facility labels, coordinate grids, and location insets.



In [ ]:
# Cell 13 — Final tables and demographic consistency checks

# Settlement table contains settlement identity and naming information only.
sett_table = settlements.drop(columns="geometry").copy()

# LGA demographic checks.
if not (
    0 <= float(lga_pop.loc[0,"population_u1_est"])
    <= float(lga_pop.loc[0,"population_u5_est"])
    <= float(lga_pop.loc[0,"population_total_est"]) * 1.01
):
    raise ValueError("Demographic hierarchy check failed.")

sex_sum = (
    float(lga_pop.loc[0,"population_male_est"])
    + float(lga_pop.loc[0,"population_female_est"])
)

consistency = pd.DataFrame([{
    "total_population_api": float(lga_pop.loc[0,"population_total_est"]),
    "male_plus_female": sex_sum,
    "difference": sex_sum - float(lga_pop.loc[0,"population_total_est"]),
    "under_1": float(lga_pop.loc[0,"population_u1_est"]),
    "under_5": float(lga_pop.loc[0,"population_u5_est"]),
}])

display(consistency.round(2))

show_cols = [
    c for c in [
        "settlement_component_id",
        "settlement_name",
        "name_match_status"
    ]
    if c in sett_table.columns
]

display(sett_table[show_cols].head(20))




In [ ]:
# Cell 14 — Export professional CSV tables and multi-panel PNG maps
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter
from adjustText import adjust_text
import mapclassify

EXPORT = STUDENT_WORK_DIR / "Final_Results"
EXPORT.mkdir(parents=True, exist_ok=True)

# Clear stale outputs from previous runs.
for old_file in EXPORT.iterdir():
    if old_file.is_file():
        old_file.unlink()

# ------------------------------------------------------------
# A. CSV EXPORTS — roads and drainage are map context only
# ------------------------------------------------------------
lga_pop.to_csv(EXPORT/"lga_population_summary.csv",index=False)

sett_table=settlements.drop(columns="geometry").copy()
sett_table.to_csv(EXPORT/"settlement_summary.csv",index=False)

health.drop(columns="geometry").to_csv(
    EXPORT/"health_facilities.csv",index=False
)

if "settlement_api_result" in globals():
    settlement_api_result.to_csv(
        EXPORT/"selected_settlement_population_query.csv",index=False
    )

density_cols=[
    c for c in [
        "settlement_component_id","settlement_name","type",
        "population_total_est","area_km2",
        "population_density_p_km2","api_status"
    ] if c in settlements.columns
]
if density_cols:
    settlements[density_cols].to_csv(
        EXPORT/"settlement_population_density.csv",index=False
    )

# ------------------------------------------------------------
# B. NATIONAL ADMINISTRATIVE CONTEXT
# ------------------------------------------------------------
admin_dir=REPO_DIR/"data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state"
admin_parts=[gpd.read_file(p).to_crs(4326) for p in sorted(admin_dir.glob("*.geojson"))]

national_lgas=gpd.GeoDataFrame(
    pd.concat(admin_parts,ignore_index=True),
    geometry="geometry",crs="EPSG:4326"
)
national_states=national_lgas.dissolve(
    by=["adm1cd","adm1nm"],as_index=False
)

MAP_CRS=selected.estimate_utm_crs()
selected_p=selected.to_crs(MAP_CRS)
national_lgas_p=national_lgas.to_crs(MAP_CRS)
settlements_p=settlements.to_crs(MAP_CRS)
roads_p=OSM_ROADS.to_crs(MAP_CRS) if len(OSM_ROADS) else OSM_ROADS.copy()
drainage_p=DRAINAGE.to_crs(MAP_CRS) if len(DRAINAGE) else DRAINAGE.copy()
health_p=health.to_crs(MAP_CRS) if len(health) else health.copy()
grid3_names_p=grid3_names.to_crs(MAP_CRS) if len(grid3_names) else grid3_names.copy()

sel_union=selected_p.geometry.union_all()
xmin,ymin,xmax,ymax=selected_p.total_bounds
span=max(xmax-xmin,ymax-ymin)
context_buffer=max(span*0.18,2500)

surrounding_p=national_lgas_p[
    national_lgas_p.geometry.intersects(sel_union.buffer(context_buffer))
].copy()

# Context extent is based on neighbouring polygons, but their boundaries are NOT drawn.
cxmin,cymin,cxmax,cymax=surrounding_p.total_bounds
cpadx=max((cxmax-cxmin)*0.05,1000)
cpady=max((cymax-cymin)*0.05,1000)
padx=max((xmax-xmin)*0.05,500)
pady=max((ymax-ymin)*0.05,500)

# ------------------------------------------------------------
# C. PROFESSIONAL SYMBOLOGY
# ------------------------------------------------------------
TYPE_COLORS={
    "urban":"#d73027",
    "city":"#d73027",
    "large":"#fc8d59",
    "town":"#fc8d59",
    "medium":"#fee08b",
    "village":"#91cf60",
    "small":"#91cf60",
    "hamlet":"#1a9850",
    "other":"#bdbdbd"
}

def settlement_color(v):
    s=str(v).strip().lower()
    for key,color in TYPE_COLORS.items():
        if key in s:
            return color
    return "#bdbdbd"

# Road hierarchy: warm colors for major roads, neutral grays for local roads.
ROAD_STYLE={
    "motorway":("#d73027",2.2),
    "trunk":("#f46d43",1.9),
    "primary":("#fdae61",1.65),
    "secondary":("#fddf8b",1.35),
    "tertiary":("#8c8c8c",1.05),
    "residential":("#bdbdbd",0.70),
    "unclassified":("#cccccc",0.60),
    "service":("#dddddd",0.45)
}

DRAIN_STYLE={
    "river":("#08519c",1.8,"-"),
    "stream":("#3182bd",1.05,"-"),
    "canal":("#6baed6",1.0,"--"),
    "drain":("#9ecae1",0.70,":")
}

def draw_roads(ax,roads):
    if len(roads)==0:
        return
    if "highway" not in roads.columns:
        roads.plot(ax=ax,color="#bdbdbd",linewidth=0.65,zorder=4)
        return
    for hw,grp in roads.groupby(roads["highway"].fillna("unclassified").astype(str)):
        color,lw=ROAD_STYLE.get(hw,("#cccccc",0.5))
        grp.plot(ax=ax,color=color,linewidth=lw,zorder=4)

def draw_drainage(ax,drain):
    if len(drain)==0:
        return
    if "waterway" not in drain.columns:
        drain.plot(ax=ax,color="#3182bd",linewidth=0.9,zorder=3)
        return
    for cls,grp in drain.groupby(drain["waterway"].fillna("stream").astype(str)):
        color,lw,ls=DRAIN_STYLE.get(cls,("#3182bd",0.8,"-"))
        grp.plot(ax=ax,color=color,linewidth=lw,linestyle=ls,zorder=3)

def present_road_handles(roads):
    if len(roads)==0 or "highway" not in roads.columns:
        return []
    present=list(dict.fromkeys(roads["highway"].dropna().astype(str)))
    order=["motorway","trunk","primary","secondary","tertiary","residential","unclassified","service"]
    handles=[]
    for hw in order:
        if hw in present:
            color,lw=ROAD_STYLE[hw]
            handles.append(Line2D([0],[0],color=color,lw=max(lw,1.2),label=hw.title()))
    return handles

def present_drain_handles(drain):
    if len(drain)==0 or "waterway" not in drain.columns:
        return []
    present=set(drain["waterway"].dropna().astype(str))
    handles=[]
    for cls in ["river","stream","canal","drain"]:
        if cls in present:
            color,lw,ls=DRAIN_STYLE[cls]
            handles.append(Line2D([0],[0],color=color,lw=max(lw,1.2),linestyle=ls,label=cls.title()))
    return handles

def add_grid(ax):
    ax.grid(True,linestyle="--",linewidth=0.45,color="#b0b0b0",alpha=0.65)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x,pos:f"{x/1000:.0f}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y,pos:f"{y/1000:.0f}"))
    ax.set_xlabel("Easting (km)",fontsize=8)
    ax.set_ylabel("Northing (km)",fontsize=8)

def add_north_arrow(ax):
    ax.annotate(
        "N",xy=(0.955,0.95),xytext=(0.955,0.84),
        xycoords="axes fraction",
        arrowprops=dict(facecolor="black",edgecolor="black",width=2.2,headwidth=8),
        ha="center",va="center",fontsize=10,fontweight="bold",zorder=30
    )

def add_scale_bar(ax):
    x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim()
    width=x1-x0
    candidates=np.array([1000,2000,5000,10000,20000,50000],dtype=float)
    target=width*0.18
    length=float(candidates[np.argmin(np.abs(candidates-target))])
    sx=x0+width*0.06
    sy=y0+(y1-y0)*0.055
    ax.plot([sx,sx+length],[sy,sy],color="black",linewidth=3,zorder=30)
    ax.plot([sx,sx],[sy-(y1-y0)*0.008,sy+(y1-y0)*0.008],color="black",linewidth=1.3,zorder=30)
    ax.plot([sx+length,sx+length],[sy-(y1-y0)*0.008,sy+(y1-y0)*0.008],color="black",linewidth=1.3,zorder=30)
    ax.text(sx+length/2,sy+(y1-y0)*0.014,f"{length/1000:g} km",
            ha="center",va="bottom",fontsize=7,color="#222222",zorder=30)

def add_lga_labels(ax,gdf):
    texts=[]
    for _,r in gdf.iterrows():
        nm=str(r.get("adm2nm","")).strip()
        if not nm:
            continue
        pt=r.geometry.representative_point()
        is_selected=str(r.get("adm2cd"))==str(LGA_ID)
        texts.append(
            ax.text(
                pt.x,pt.y,nm,
                fontsize=7.3 if is_selected else 6.1,
                color="#17365d" if is_selected else "#4d4d4d",
                fontweight="bold" if is_selected else "normal",
                ha="center",va="center",zorder=15
            )
        )
    if texts:
        adjust_text(
            texts,ax=ax,
            expand=(1.05,1.15),
            force_text=(0.35,0.45),
            ensure_inside_axes=True
        )

def add_settlement_labels(ax,max_labels=100):
    if len(grid3_names_p)==0 or "set_name" not in grid3_names_p.columns:
        return
    pts=grid3_names_p.copy()
    if len(pts)>max_labels:
        pts=pts.head(max_labels)
    texts=[]
    for _,r in pts.iterrows():
        nm=str(r.get("set_name","")).strip()
        if nm and nm.lower()!="nan":
            texts.append(
                ax.text(
                    r.geometry.x,r.geometry.y,nm,
                    fontsize=5.1,color="#1b5e20",
                    ha="center",va="center",zorder=12
                )
            )
    if texts:
        adjust_text(
            texts,ax=ax,
            expand=(1.02,1.10),
            force_text=(0.25,0.35),
            ensure_inside_axes=True
        )

def add_health_labels(ax,max_labels=70):
    if len(health_p)==0 or "facility_name" not in health_p.columns:
        return
    texts=[]
    xs=[]; ys=[]
    for _,r in health_p.head(max_labels).iterrows():
        nm=str(r.get("facility_name","")).strip()
        if nm and nm.lower()!="nan":
            xs.append(r.geometry.x); ys.append(r.geometry.y)
            texts.append(
                ax.text(
                    r.geometry.x,r.geometry.y,nm,
                    fontsize=4.8,color="#8b1a1a",
                    ha="left",va="bottom",zorder=14
                )
            )
    if texts:
        adjust_text(
            texts,x=xs,y=ys,ax=ax,
            expand=(1.04,1.12),
            force_text=(0.35,0.45),
            arrowprops=dict(arrowstyle="-",color="#8b1a1a",lw=0.35,alpha=0.55),
            ensure_inside_axes=True
        )

def draw_density(ax,valid_gdf,nodata_gdf):
    # No settlement polygon outlines: fill only.
    if len(nodata_gdf):
        nodata_gdf.plot(
            ax=ax,facecolor="#eeeeee",edgecolor="none",
            linewidth=0,zorder=2
        )

    if len(valid_gdf):
        vals=pd.to_numeric(valid_gdf["population_density_p_km2"],errors="coerce")
        valid_gdf=valid_gdf.loc[vals.notna()].copy()
        valid_gdf["population_density_p_km2"]=vals.loc[vals.notna()]
        unique_n=int(valid_gdf["population_density_p_km2"].nunique())

        if unique_n>=2:
            valid_gdf.plot(
                column="population_density_p_km2",
                ax=ax,cmap="YlOrRd",scheme="Quantiles",
                k=min(5,unique_n),
                edgecolor="none",linewidth=0,
                legend=True,
                legend_kwds={
                    "title":"Population density\n(persons/km²)",
                    "loc":"lower right",
                    "fontsize":6.5,
                    "title_fontsize":7
                },
                zorder=2
            )
        else:
            valid_gdf.plot(
                ax=ax,facecolor="#f03b20",
                edgecolor="none",linewidth=0,zorder=2
            )

# ------------------------------------------------------------
# D. MULTI-PANEL MAP
# ------------------------------------------------------------
fig=plt.figure(figsize=(18,14),constrained_layout=False)
gs=fig.add_gridspec(
    2,2,left=0.055,right=0.97,top=0.90,bottom=0.06,
    wspace=0.12,hspace=0.16
)
ax1=fig.add_subplot(gs[0,0])
ax2=fig.add_subplot(gs[0,1])
ax3=fig.add_subplot(gs[1,0])
ax4=fig.add_subplot(gs[1,1])

# PANEL A — selected LGA + neighbouring LGA labels only.
# Surrounding LGA boundaries are intentionally NOT drawn.
selected_p.plot(
    ax=ax1,facecolor="#f7f7f7",edgecolor="#252525",
    linewidth=1.8,zorder=1
)
draw_drainage(ax1,drainage_p)
draw_roads(ax1,roads_p)
add_lga_labels(ax1,surrounding_p)

ax1.set_xlim(cxmin-cpadx,cxmax+cpadx)
ax1.set_ylim(cymin-cpady,cymax+cpady)
ax1.set_title("A. LGA Context, Road Hierarchy and Drainage",fontsize=12,fontweight="bold")
add_grid(ax1); add_north_arrow(ax1); add_scale_bar(ax1)

context_handles=[
    Patch(facecolor="#f7f7f7",edgecolor="#252525",label="Selected LGA")
]
context_handles+=present_road_handles(roads_p)
context_handles+=present_drain_handles(drainage_p)
ax1.legend(
    handles=context_handles,
    loc="lower right",fontsize=6.4,frameon=True,
    title="Legend",title_fontsize=8,ncol=2
)

# PANEL B — settlement type fills, no settlement outlines.
selected_p.plot(
    ax=ax2,facecolor="white",edgecolor="#252525",
    linewidth=1.5,zorder=1
)

sett_type_values=[]
if "type" in settlements_p.columns:
    for typ,grp in settlements_p.groupby(
        settlements_p["type"].fillna("Other").astype(str)
    ):
        color=settlement_color(typ)
        grp.plot(
            ax=ax2,facecolor=color,edgecolor="none",
            linewidth=0,alpha=0.82,zorder=2
        )
        sett_type_values.append((typ,color))
else:
    settlements_p.plot(
        ax=ax2,facecolor="#fee08b",
        edgecolor="none",linewidth=0,alpha=0.82,zorder=2
    )

draw_drainage(ax2,drainage_p)
draw_roads(ax2,roads_p)

if len(health_p):
    health_p.plot(
        ax=ax2,color="#b2182b",marker="P",markersize=25,
        edgecolor="white",linewidth=0.35,zorder=10
    )

add_settlement_labels(ax2,max_labels=100)
add_health_labels(ax2,max_labels=70)

ax2.set_xlim(xmin-padx,xmax+padx)
ax2.set_ylim(ymin-pady,ymax+pady)
ax2.set_title("B. Settlement Type, Roads, Drainage and Health Facilities",fontsize=12,fontweight="bold")
add_grid(ax2); add_north_arrow(ax2); add_scale_bar(ax2)

type_handles=[
    Patch(facecolor=c,edgecolor="none",label=str(t))
    for t,c in sorted(set(sett_type_values),key=lambda x:str(x[0]))
]
type_handles += [
    Line2D([0],[0],marker="P",color="none",markerfacecolor="#b2182b",
           markeredgecolor="white",markersize=8,label="Health facility")
]
type_handles += present_road_handles(roads_p)
type_handles += present_drain_handles(drainage_p)
ax2.legend(
    handles=type_handles,
    loc="lower right",fontsize=5.8,frameon=True,
    title="Legend",title_fontsize=7,ncol=2
)

# PANEL C — settlement population-density fills, no settlement outlines.
selected_p.plot(
    ax=ax3,facecolor="white",edgecolor="#252525",
    linewidth=1.5,zorder=1
)

valid_density=settlements_p[
    pd.to_numeric(
        settlements_p.get("population_density_p_km2"),
        errors="coerce"
    ).notna()
].copy() if "population_density_p_km2" in settlements_p.columns else settlements_p.iloc[0:0].copy()

nodata_density=settlements_p.drop(valid_density.index)

draw_density(ax3,valid_density,nodata_density)
draw_drainage(ax3,drainage_p)
draw_roads(ax3,roads_p)
add_settlement_labels(ax3,max_labels=100)

ax3.set_xlim(xmin-padx,xmax+padx)
ax3.set_ylim(ymin-pady,ymax+pady)
ax3.set_title("C. Settlement Population Density — WorldPop API",fontsize=12,fontweight="bold")
add_grid(ax3); add_north_arrow(ax3); add_scale_bar(ax3)

# PANEL D — Nigeria/State insets and study summary.
ax4.axis("off")
ax4.set_title("D. Location Insets and Study Summary",fontsize=12,fontweight="bold",pad=10)

ax_ng=inset_axes(ax4,width="44%",height="45%",loc="upper left",borderpad=1.0)
national_states.plot(
    ax=ax_ng,facecolor="#f0f0f0",edgecolor="#969696",linewidth=0.45
)
state_hit=national_states[
    national_states["adm1cd"].astype(str).eq(str(STATE_ID))
]
if len(state_hit):
    state_hit.plot(
        ax=ax_ng,facecolor="#3182bd",edgecolor="#08519c",linewidth=0.8
    )
ax_ng.set_title("Selected State/FCT in Nigeria",fontsize=8,fontweight="bold")
ax_ng.set_axis_off()

ax_st=inset_axes(ax4,width="44%",height="45%",loc="upper right",borderpad=1.0)
state_inset=state_gdf.to_crs(4326)
state_inset.plot(
    ax=ax_st,facecolor="#f2f2f2",edgecolor="#969696",linewidth=0.45
)
selected.to_crs(4326).plot(
    ax=ax_st,facecolor="#fd8d3c",edgecolor="#7f2704",linewidth=1.0
)
ax_st.set_title(f"{SELECTED_LGA} in {SELECTED_STATE}",fontsize=8,fontweight="bold")
ax_st.set_axis_off()

lga_area_km2=float(selected_p.geometry.area.iloc[0]/1_000_000)
total_pop=float(lga_pop.loc[0,"population_total_est"])
overall_density=total_pop/lga_area_km2 if lga_area_km2>0 else np.nan
n_named=int(
    settlements["settlement_name"].notna().sum()
    if "settlement_name" in settlements.columns else 0
)
n_density=int(
    settlements["population_density_p_km2"].notna().sum()
    if "population_density_p_km2" in settlements.columns else 0
)

summary=(
    f"Study LGA: {SELECTED_LGA}, {SELECTED_STATE}\n"
    f"LGA code: {LGA_ID}\n"
    f"Area: {lga_area_km2:,.1f} km²\n"
    f"Estimated population ({WORLDPOP_YEAR}): {total_pop:,.0f}\n"
    f"Overall density: {overall_density:,.1f} persons/km²\n"
    f"Male: {float(lga_pop.loc[0,'population_male_est']):,.0f}   "
    f"Female: {float(lga_pop.loc[0,'population_female_est']):,.0f}\n"
    f"Under 1: {float(lga_pop.loc[0,'population_u1_est']):,.0f}   "
    f"Under 5: {float(lga_pop.loc[0,'population_u5_est']):,.0f}\n"
    f"Settlement polygons: {len(settlements):,}\n"
    f"Named settlements: {n_named:,}\n"
    f"Density results: {n_density:,}\n"
    f"Health facilities: {len(health):,}\n"
    f"Drainage features: {len(DRAINAGE):,}\n\n"
    "Sources\n"
    "Administrative boundaries: UN SALB Nigeria\n"
    "Settlement extents/names: GRID3\n"
    "Population/density: WorldPop API v2\n"
    "Roads and drainage: OpenStreetMap\n"
    "Health facilities: GRID3/NHFR"
)

ax4.text(
    0.03,0.42,summary,transform=ax4.transAxes,
    va="top",ha="left",fontsize=9,linespacing=1.35,
    bbox=dict(
        boxstyle="round,pad=0.7",
        facecolor="#fafafa",edgecolor="#969696",linewidth=0.8
    )
)

fig.suptitle(
    f"Population, Settlements, Health Facilities, Roads and Drainage\n"
    f"{SELECTED_LGA}, {SELECTED_STATE}, Nigeria",
    fontsize=17,fontweight="bold",y=0.965
)

fig.text(
    0.5,0.018,
    "Population values are modelled estimates; GRID3 settlement extents are mapped extents, "
    "not official administrative boundaries.",
    ha="center",va="bottom",fontsize=7,color="#555555"
)

MULTI_PNG=EXPORT/"01_professional_multipanel_map.png"
fig.savefig(
    MULTI_PNG,dpi=300,bbox_inches="tight",facecolor="white"
)
plt.close(fig)

# ------------------------------------------------------------
# E. STANDALONE POPULATION-DENSITY MAP
# ------------------------------------------------------------
fig,ax=plt.subplots(figsize=(11,10))
selected_p.plot(
    ax=ax,facecolor="white",edgecolor="#252525",
    linewidth=1.5,zorder=1
)
draw_density(ax,valid_density,nodata_density)
draw_drainage(ax,drainage_p)
draw_roads(ax,roads_p)
add_settlement_labels(ax,max_labels=120)

ax.set_xlim(xmin-padx,xmax+padx)
ax.set_ylim(ymin-pady,ymax+pady)
ax.set_title(
    f"Settlement Population Density\n{SELECTED_LGA}, {SELECTED_STATE}",
    fontsize=14,fontweight="bold"
)
add_grid(ax); add_north_arrow(ax); add_scale_bar(ax)

DENSITY_PNG=EXPORT/"02_population_density_map.png"
fig.savefig(
    DENSITY_PNG,dpi=300,bbox_inches="tight",facecolor="white"
)
plt.close(fig)

print("Professional outputs saved:")
for p in sorted(EXPORT.iterdir()):
    print(" -",p.name)

# ------------------------------------------------------------
# F. ZIP ONLY CSV + PNG
# ------------------------------------------------------------
ZIP_DRIVE=STUDENT_WORK_DIR/f"{safe(SELECTED_STATE)}_{safe(SELECTED_LGA)}_professional_maps_csv.zip"
if ZIP_DRIVE.exists():
    ZIP_DRIVE.unlink()

files_to_zip=[
    p.name for p in EXPORT.iterdir()
    if p.suffix.lower() in {".csv",".png"}
]

subprocess.run(
    ["zip","-q",str(ZIP_DRIVE),*files_to_zip],
    cwd=EXPORT,check=True
)

print("ZIP:",ZIP_DRIVE)
from google.colab import files
files.download(str(ZIP_DRIVE))



## Completed

You have selected an LGA, mapped GRID3 settlements and names, estimated population and demographics, added health-facility and OpenStreetMap road context, mapped settlement population density, and exported professional CSV and PNG outputs.

